# Step 7 — Making Predictions

## Purpose
Use the trained model to make predictions on new data, and demonstrate three real prediction types the app supports:

1. **Classification** — given feature values, predict the class
2. **Regression** — predict a continuous numeric value
3. **Time-series forecast** — project a numeric value forward by N months

## Project reference
Real implementation: [`../trainer.py`](../trainer.py), [`../reasoner.py`](../reasoner.py), and the `_run_forecast` / `_run_regression` helpers in [`../app.py`](../app.py).

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../..'))   # project root from eda/notebooks/

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import data_loader as dl

raw   = dl.generate_synthetic_data(n_rows=2000, seed=42)
clean = dl.clean_data(raw, target_col='sales_category')
X     = clean.drop(columns=['sales_category'])
le    = LabelEncoder()
y     = le.fit_transform(clean['sales_category'].astype(str))
rf    = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X, y)
print('Trained classifier ready.')

## 7.1 Prediction Type 1 — Classification

Predict the `sales_category` for a hypothetical new sale.

In [ ]:
# Build a single-row input using median defaults so it's a realistic record
new_row = pd.DataFrame([X.median(numeric_only=True).to_dict()])
# Override the fields the user knows
new_row['quantity']     = 8
new_row['discount_pct'] = 10
new_row['unit_price']   = 25000

# Predict
pred_idx  = rf.predict(new_row)[0]
pred_prob = rf.predict_proba(new_row)[0]
pred_lbl  = le.inverse_transform([pred_idx])[0]

print(f'Predicted sales_category: {pred_lbl!r}')
print(f'Confidence per class:')
for cls, p in zip(le.classes_, pred_prob):
    print(f'  {cls:8} → {p:.1%}')

## 7.2 Prediction Type 2 — Regression

Predict a continuous numeric value (here: `net_sales`) given other features.

In [ ]:
# Build a regression problem: predict net_sales from everything else numeric
num_X = clean.select_dtypes(include=[np.number]).drop(columns=['net_sales'])
num_y = clean['net_sales']

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(num_X, num_y)

new_inp           = num_X.median().to_dict()
new_inp['quantity']     = 5
new_inp['unit_price']   = 30000
new_inp['discount_pct'] = 8

pred_val = rf_reg.predict(pd.DataFrame([new_inp]))[0]
r2       = rf_reg.score(num_X, num_y)

print(f'Predicted net_sales: Rs {pred_val:,.2f}')
print(f'Model R² on training data: {r2:.1%}')
print('\nTop 5 features driving this prediction:')
imp = pd.Series(rf_reg.feature_importances_, index=num_X.columns).sort_values(ascending=False).head(5)
for f, v in imp.items():
    print(f'  {f:18} → weight {v:.3f}')

## 7.3 Prediction Type 3 — Time-series forecast

Project `net_sales` forward by 6 months using a linear trend on monthly-aggregated data.

In [ ]:
ts = raw[['date', 'net_sales']].copy()
ts['date'] = pd.to_datetime(ts['date'])
monthly    = ts.set_index('date').resample('ME')['net_sales'].mean().dropna()

X_time = np.arange(len(monthly)).reshape(-1, 1)
y_time = monthly.values.astype(float)
trend  = LinearRegression().fit(X_time, y_time)

future_X = np.arange(len(monthly), len(monthly) + 6).reshape(-1, 1)
future_y = trend.predict(future_X)
future_d = pd.date_range(monthly.index[-1], periods=7, freq='ME')[1:]

trend_dir = 'increasing 📈' if trend.coef_[0] > 0 else 'decreasing 📉'
print(f'Historical monthly average:  Rs {monthly.mean():,.0f}')
print(f'Trend direction:             {trend_dir}')
print(f'\nNext 6 months forecast:')
for d, v in zip(future_d, future_y):
    print(f'  {d.strftime("%Y-%m")}  →  Rs {v:,.0f}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly.index, monthly.values, label='historical', color='#7FA98B', marker='o')
ax.plot(future_d, future_y, label='forecast', color='#E07856', marker='o', linestyle='--')
ax.legend(); ax.set_title('net_sales — monthly average + 6-month forecast')
ax.set_ylabel('Rs'); plt.tight_layout(); plt.show()

## Summary of Step 7 — End-to-End Predictions

We demonstrated all three prediction types the app supports:

| Prediction type | Library used | Use case |
|---|---|---|
| Classification | `RandomForestClassifier` | "Will this sale be high/medium/low?" |
| Regression | `RandomForestRegressor` | "What will the exact net_sales be?" |
| Time-series forecast | `LinearRegression` | "What will net_sales look like next 6 months?" |

Every prediction comes with:
- A confidence score (classification) or R² (regression)
- A ranking of the features that drove it
- A plain-English explanation in the live app's chat panel

## End of EDA pipeline

Steps 1 → 7 walked through the complete machine-learning lifecycle:

1. Load & inspect
2. Clean
3. Explore (EDA proper)
4. Preprocess & engineer features
5. Train models
6. Evaluate
7. Predict

Every step in these notebooks corresponds to actual code in the deployed app under [`../`](../).